# MediaAgent Eval

**Сравнение архитектур AI-агентов на вопросах о музыке, фильмах и сериалах**

В работе сравниваются следующие конфигурации:

1. **Baseline ReAct** - базовый агент с инструментами поиска информации.
2. **LangGraph Agent** - тот же агент, реализованный с использованием LangGraph.
3. **Critic** - агент с дополнительной проверкой ответов по источникам.
4. **Self-Consistency (Voting)** - получение нескольких независимых ответов и голосование.
5. **Cascade** - обращение к более сильной модели при недостаточном согласии.
6. **Strong Model Baseline** - сильная модель без дополнительных агентных паттернов.



In [1]:
import os
import json
import time
import re
from pathlib import Path
from collections import Counter

import httpx
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from pydantic import BaseModel, Field

In [2]:
load_dotenv(Path.cwd() / ".env")
API_KEY = os.getenv("OPENROUTER_API_KEY")

if not API_KEY:
    raise ValueError("Не найден OPENROUTER_API_KEY в .env")

CHAT_URL = "https://openrouter.ai/api/v1/chat/completions"
CHEAP_MODEL = "openai/gpt-4o-mini"
STRONG_MODEL = "anthropic/claude-sonnet-4.6"


DATA_PATH = Path("data/questions.jsonl")
with open(DATA_PATH, "r", encoding="utf-8") as f:
    questions = [json.loads(line) for line in f if line.strip()]

df = pd.DataFrame(questions)
print(f"Всего вопросов: {len(df)}")
print(f"Столбцы: {df.columns.tolist()}")
display(df.head())

Всего вопросов: 50
Столбцы: ['id', 'source', 'category', 'period', 'question', 'answer', 'aliases', 'url', 'evidence']


,id,source,category,period,question,answer,aliases,url,evidence
0,music-2026-01,culturecheck,music,2026,Which album won Album of the Year at the 2026 ...,DeBÍ TiRAR MáS FOToS,"[Debi Tirar Mas Fotos, Debí Tirar Más Fotos]",https://www.grammy.com/awards/68th-annual-gram...,The Recording Academy lists Bad Bunny's DeBÍ T...
1,music-2026-02,culturecheck,music,2026,Which song won Record of the Year at the 2026 ...,luther,[Luther],https://www.grammy.com/awards/68th-annual-gram...,The 2026 Grammy Record of the Year went to lut...
2,music-2026-03,culturecheck,music,2026,Which song won Song of the Year at the 2026 Gr...,WILDFLOWER,[Wildflower],https://www.grammy.com/awards/68th-annual-gram...,The Recording Academy credits Billie Eilish an...
3,music-2026-04,culturecheck,music,2026,Who won Best New Artist at the 2026 Grammy Awa...,Olivia Dean,[],https://www.grammy.com/awards/68th-annual-gram...,Olivia Dean is listed as the Best New Artist w...
4,music-2026-05,culturecheck,music,2026,Which song earned Lola Young the 2026 Grammy f...,Messy,[],https://www.grammy.com/awards/68th-annual-gram...,Lola Young received the 2026 Best Pop Solo Per...


In [6]:
print("Пропуски по столбцам:")
print(df.isna().sum())
print()
print("Повторяющиеся ID:", df["id"].duplicated().sum())
print("Повторяющиеся вопросы:", df["question"].str.strip().str.lower().duplicated().sum())
print()
print("Распределение по категориям и периодам:")
display(pd.crosstab(df["category"], df["period"], margins=True))

Пропуски по столбцам:
id          0
source      0
category    0
period      0
question    0
answer      0
aliases     0
url         0
evidence    0
dtype: int64

Повторяющиеся ID: 0
Повторяющиеся вопросы: 0

Распределение по категориям и периодам:


period,2026,before_2024,All
category,,,
film,10,10,20
music,10,10,20
series,5,5,10
All,25,25,50


#### HTTP-счётчик запросов к моделям:

In [7]:
CALLS = []
FIRST = {}
TAG = {"now": "без метки"}

def on_response(response):
    response.read()

    request = json.loads(response.request.content or b"{}")
    data = response.json()
    usage = data.get("usage") or {}
    FIRST.setdefault(TAG["now"], request)
    CALLS.append({ "tag": TAG["now"], "model": request.get("model", "?"),
        "messages": len(request.get("messages", [])),
        "tools": len(request.get("tools", [])),
        "prompt_tokens": usage.get("prompt_tokens", 0),
        "completion_tokens": usage.get("completion_tokens", 0),
        "cost": usage.get("cost") or 0, "status_code": response.status_code
    })

#### Общий HTTP-клиента:

In [9]:
HTTP = httpx.Client(event_hooks={"response": [on_response]}, timeout=180.0)

In [10]:
HEADERS = { "Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"}

def chat(messages, model=CHEAP_MODEL, tools=None, temperature=0.0):
    body = { "model": model, "messages": messages, "temperature": temperature}

    if tools:
        body["tools"] = tools

    for attempt in range(5):
        response = HTTP.post( CHAT_URL, json=body, headers=HEADERS)

        if response.status_code == 200:
            return response.json()["choices"][0]["message"]

        if response.status_code not in (429, 500, 502, 503, 504):
            response.raise_for_status()

        time.sleep(2 * (attempt + 1))

    raise RuntimeError( f"HTTP {response.status_code}: {response.text[:200]}")

In [11]:
TAG["now"] = "test"
messages = [ {"role": "user", "content": "What is 2 + 2? Reply with only the number."}]
answer = chat(messages)

print("Ответ модели:", answer["content"])
print("\nПоследний запрос:")
print(CALLS[-1])

Ответ модели: 4

Последний запрос:
{'tag': 'test', 'model': 'openai/gpt-4o-mini', 'messages': 1, 'tools': 0, 'prompt_tokens': 21, 'completion_tokens': 1, 'cost': 3.75e-06, 'status_code': 200}


## Tools

#### Подключение к англоязычной Википедии:

In [13]:
WIKI = "https://en.wikipedia.org/w/api.php"
WIKI_HEADERS = { "User-Agent": "MediaAgentEval/0.1 (educational project)"}

def wiki(params, attempts=3):
    for attempt in range(attempts):
        r = requests.get( WIKI, params={**params, "format": "json"}, headers=WIKI_HEADERS, timeout=15)
        
        if r.status_code == 200:
            data = r.json()
            if "error" not in data:
                return data["query"]

        time.sleep(1 + attempt)

    raise RuntimeError(f"Wikipedia API error: HTTP {r.status_code}")

#### поиск информации в Википедии:

In [14]:
def web_search(query: str) -> str:
    hits = wiki({ "action": "query",  "list": "search", "srsearch": query, "srlimit": 3}).get("search", [])

    if not hits:
        return "Nothing found"

    pages = wiki({ "action": "query",  "prop": "extracts", "explaintext": 1, "exintro": 1, "titles": hits[0]["title"] }).get("pages", {})
    intro = " ".join( next(iter(pages.values()), {}).get("extract", "").split())[:1200]
    other_titles = ", ".join(h["title"] for h in hits[1:])

    return f"[{hits[0]['title']}] {intro} | Other articles: {other_titles}"

#### поиск информации внутри конкретной статьи Википедии:

In [16]:
def page_find(title: str, keywords: str) -> str:
    pages = wiki({ "action": "query", "prop": "extracts", "explaintext": 1, "titles": title, "redirects": 1}).get("pages", {})
    text = next(iter(pages.values()), {}).get("extract", "")

    if not text:
        return "No such page"

    words = [ w for w in re.findall(r"\w+", keywords.lower()) if len(w) > 2]
    lines = [ line.strip() for line in text.splitlines() if line.strip()]
    hits = [  line for line in lines
        if sum(w in line.lower() for w in words) >= max(1, (len(words) + 1) // 2)]

    return "\n".join(line[:350] for line in hits[:5]) or "Keywords not found"

In [19]:
result = page_find( title="Interstellar (film)", keywords="Hans Zimmer score")
print(result)

Hans Zimmer, who scored Nolan's The Dark Knight Trilogy and Inception (2010), returned to score Interstellar. Nolan chose not to provide Zimmer with a script or any plot details but instead gave him a single page that told the story of a father leaving his child for work. It was through this connection that Zimmer created the early stages of the In


**регистрация инструментов:**

In [20]:
class SearchArgs(BaseModel):
    query: str = Field(description="Search query: artist, film, series, or event")


class PageFindArgs(BaseModel):
    title: str = Field(description="Exact Wikipedia article title")
    keywords: str = Field(description="Keywords to search within the article")


TOOLS = {}

def register(fn, args_model, description):
    TOOLS[fn.__name__] = {
        "fn": fn,
        "args": args_model,
        "schema": {
            "type": "function",
            "function": {
                "name": fn.__name__,
                "description": description,
                "parameters": args_model.model_json_schema()
            }
        }
    }


register(web_search, SearchArgs, "Search English Wikipedia for relevant articles")
register(page_find, PageFindArgs, "Find information inside a Wikipedia article")

print("Инструментов зарегистрировано:", len(TOOLS))

Инструментов зарегистрировано: 2
